# Notebook 07 — GeoZarr

**What you will learn:**
- What GeoZarr is and what problem it solves
- How spatial metadata (CRS, grid_mapping) is encoded in Zarr `.zattrs` JSON
- How to write a GeoZarr-compliant store and validate it with GDAL and rioxarray
- How to write the GeoZarr store to S3

---

## Background: What is GeoZarr?

**GeoZarr is not a library.** It is a set of conventions — a specification — that describes how to store CRS and spatial metadata inside a Zarr store's `.zattrs` JSON files.

**The problem it solves:** Plain Zarr knows nothing about geography. If you save an xarray Dataset with spatial data to Zarr using `ds.to_zarr()`, the resulting store contains pixel values and axis labels — but a tool like GDAL or QGIS cannot tell where those pixels are on Earth, or what projection they use, unless the metadata is encoded in a standard way.

**How it works:** GeoZarr follows the [CF (Climate and Forecast) Conventions](https://cfconventions.org/) for grid mapping. Each data variable must have a `grid_mapping` attribute pointing to a coordinate variable that contains the CRS definition as a WKT string. GDAL ≥ 3.8 reads this natively.

```
sentinel2.zarr/
  red/
    .zarray       ← shape, dtype, chunks
    .zattrs       ← {"grid_mapping": "spatial_ref", ...}   ← KEY: points to CRS variable
    0.0  0.1 ...
  spatial_ref/
    .zarray
    .zattrs       ← {"crs_wkt": "PROJCRS[...]", "grid_mapping_name": "..."}
```

In [1]:
import sys
sys.path.insert(0, "..")

import xarray as xr
import rioxarray
import zarr
import json
import os
import subprocess
from utils.zarr_helpers import get_s3_store, verify_roundtrip

ZARR_ENRICHED = "../data/sentinel2_enriched.zarr"
ZARR_GEOZARR  = "../data/sentinel2_geozarr.zarr"

ds = xr.open_zarr(ZARR_ENRICHED)
print(ds)

<xarray.Dataset> Size: 217MB
Dimensions:      (time: 3, y: 2788, x: 2158)
Coordinates:
  * time         (time) datetime64[ns] 24B 2023-06-04T16:02:09.577000 ... 202...
  * x            (x) float64 17kB 4.355e+05 4.355e+05 ... 4.786e+05 4.786e+05
  * y            (y) float64 22kB 4.428e+06 4.428e+06 ... 4.372e+06 4.372e+06
Data variables:
    blue         (time, y, x) uint16 36MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    green        (time, y, x) uint16 36MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    ndvi         (time, y, x) float32 72MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    nir          (time, y, x) uint16 36MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    red          (time, y, x) uint16 36MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    spatial_ref  int64 8B ...


## Step 1: Check what is currently in .zattrs (without CRS)

After a plain `to_zarr()` without rioxarray, the `.zattrs` for a band variable contains only xarray's internal metadata — no `grid_mapping`, no CRS WKT. GDAL cannot locate the data spatially.

In [2]:
z = zarr.open(ZARR_ENRICHED, mode="r")

print("=== red/.zattrs (plain zarr, no CRS) ===")
red_attrs = dict(z["red"].attrs)
print(json.dumps(red_attrs, indent=2))

print("\n'grid_mapping' present:", "grid_mapping" in red_attrs)
print("'crs_wkt' present:     ", any("crs" in k.lower() for k in red_attrs))

=== red/.zattrs (plain zarr, no CRS) ===
{
  "_ARRAY_DIMENSIONS": [
    "time",
    "y",
    "x"
  ],
  "grid_mapping": "spatial_ref",
  "nodata": 0
}

'grid_mapping' present: True
'crs_wkt' present:      False


## Step 2: Attach CRS with rioxarray and write GeoZarr store

`rio.write_crs("EPSG:32610", grid_mapping_name="spatial_ref")` adds:
1. A `grid_mapping` attribute to each data variable, pointing to `"spatial_ref"`
2. A new coordinate variable `spatial_ref` whose `.zattrs` contains the full CRS WKT

This is the GeoZarr / CF convention.

In [3]:
# Attach spatial dims and CRS
ds_geo = ds.rio.set_spatial_dims(x_dim="x", y_dim="y")
ds_geo = ds_geo.rio.write_crs("EPSG:32618", grid_mapping_name="spatial_ref")

print("Variables now:", list(ds_geo.data_vars))
print("Coordinates now:", list(ds_geo.coords))

# Save to a new zarr store
ds_geo.to_zarr(ZARR_GEOZARR, mode="w")
print(f"\nGeoZarr store written to {ZARR_GEOZARR}")

Variables now: ['blue', 'green', 'ndvi', 'nir', 'red']
Coordinates now: ['spatial_ref', 'time', 'x', 'y']

GeoZarr store written to ../data/sentinel2_geozarr.zarr


## Step 3: Inspect the .zattrs JSON on disk

Now we can see exactly what the GeoZarr convention looks like on disk.

In [4]:
z_geo = zarr.open(ZARR_GEOZARR, mode="r")

print("=== red/.zattrs (GeoZarr) ===")
geo_red_attrs = dict(z_geo["red"].attrs)
print(json.dumps(geo_red_attrs, indent=2))

print("\n=== spatial_ref/.zattrs (the CRS variable) ===")
if "spatial_ref" in z_geo:
    sr_attrs = dict(z_geo["spatial_ref"].attrs)
    # Print all keys; truncate the long WKT string
    for k, v in sr_attrs.items():
        val = str(v)
        print(f"  {k}: {val[:120]}{'...' if len(val) > 120 else ''}")
else:
    print("  spatial_ref not found — check rioxarray version")

=== red/.zattrs (GeoZarr) ===
{
  "_ARRAY_DIMENSIONS": [
    "time",
    "y",
    "x"
  ],
  "grid_mapping": "spatial_ref",
  "nodata": 0
}

=== spatial_ref/.zattrs (the CRS variable) ===
  _ARRAY_DIMENSIONS: []
  crs_wkt: PROJCS["WGS 84 / UTM zone 18N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG"...
  false_easting: 500000.0
  false_northing: 0.0
  geographic_crs_name: WGS 84
  grid_mapping_name: transverse_mercator
  horizontal_datum_name: World Geodetic System 1984
  inverse_flattening: 298.257223563
  latitude_of_projection_origin: 0.0
  longitude_of_central_meridian: -75.0
  longitude_of_prime_meridian: 0.0
  prime_meridian_name: Greenwich
  projected_crs_name: WGS 84 / UTM zone 18N
  reference_ellipsoid_name: WGS 84
  scale_factor_at_central_meridian: 0.9996
  semi_major_axis: 6378137.0
  semi_minor_axis: 6356752.314245179
  spatial_ref: PROJCS["WGS 84 / UTM zone 18N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.25722

## Step 4: Validate with GDAL

GDAL ≥ 3.8 can read Zarr stores directly via the `ZARR:` driver. If the `grid_mapping` + `spatial_ref` attributes are correct, `gdalinfo` will report the CRS.

This is the key interoperability test: **we are reading our xarray output with a completely different tool that knows nothing about Python or xarray.**

In [5]:
gdal_path = f"ZARR:{os.path.abspath(ZARR_GEOZARR)}:/red"
print("GDAL path:", gdal_path)

result = subprocess.run(
    ["gdalinfo", gdal_path],
    capture_output=True,
    text=True,
)

if result.returncode == 0:
    print(result.stdout[:2000])
    crs_found = "UTM zone 18N" in result.stdout or "32618" in result.stdout
    print(f"\nCRS detected by GDAL: {crs_found}")
else:
    print("gdalinfo returned error:")
    print(result.stderr)
    print("\nInstall GDAL: conda install -c conda-forge gdal")

GDAL path: ZARR:/home/jovyan/work/data/sentinel2_geozarr.zarr:/red
Driver: Zarr/Zarr
Files: /home/jovyan/work/data/sentinel2_geozarr.zarr/red/.zarray
Size is 2158, 2788
Coordinate System is:
PROJCRS["WGS 84 / UTM zone 18N",
    BASEGEOGCRS["WGS 84",
        DATUM["World Geodetic System 1984",
            ELLIPSOID["WGS 84",6378137,298.257223563,
                LENGTHUNIT["metre",1]]],
        PRIMEM["Greenwich",0,
            ANGLEUNIT["degree",0.0174532925199433]],
        ID["EPSG",4326]],
    CONVERSION["UTM zone 18N",
        METHOD["Transverse Mercator",
            ID["EPSG",9807]],
        PARAMETER["Latitude of natural origin",0,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8801]],
        PARAMETER["Longitude of natural origin",-75,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8802]],
        PARAMETER["Scale factor at natural origin",0.9996,
            SCALEUNIT["unity",1],
            ID["EPSG",8805]],
        PARAM

## Step 5: Validate with rioxarray as a fresh reader

`rioxarray.open_rasterio()` opens a Zarr array directly (without xarray's `open_zarr`). It reads the `grid_mapping` attribute and resolves the CRS from the `spatial_ref` variable — no Python context from earlier in this notebook.

In [6]:
red_path = os.path.join(os.path.abspath(ZARR_GEOZARR), "red")

try:
    da_fresh = rioxarray.open_rasterio(red_path, engine="zarr")
    print("Opened with rioxarray.open_rasterio")
    print("CRS:", da_fresh.rio.crs)
    print("EPSG:", da_fresh.rio.crs.to_epsg() if da_fresh.rio.crs else None)
    print("Shape:", da_fresh.shape)
except Exception as e:
    print(f"open_rasterio failed: {e}")
    print("Falling back to xarray.open_zarr + rio accessor...")
    # decode_coords="all" promotes spatial_ref from a data variable to a
    # coordinate, which lets rioxarray find and parse the CRS WKT.
    ds_fresh = xr.open_zarr(ZARR_GEOZARR, decode_coords="all")
    ds_fresh = ds_fresh.rio.set_spatial_dims(x_dim="x", y_dim="y")
    print("CRS:", ds_fresh.rio.crs)

Opened with rioxarray.open_rasterio
CRS: None
EPSG: None
open_rasterio failed: 'Dataset' object has no attribute 'shape'
Falling back to xarray.open_zarr + rio accessor...
CRS: EPSG:32618


## Compare .zattrs: plain Zarr vs GeoZarr

In [7]:
print("=== red/.zattrs COMPARISON ===")
print()
print("Plain Zarr (no CRS):")
plain_keys = set(dict(zarr.open(ZARR_ENRICHED, mode="r")["red"].attrs).keys())
for k in sorted(plain_keys):
    print(f"  {k}")

print()
print("GeoZarr (with CRS):")
geo_keys = set(dict(z_geo["red"].attrs).keys())
for k in sorted(geo_keys):
    mark = " <-- ADDED" if k not in plain_keys else ""
    print(f"  {k}{mark}")

=== red/.zattrs COMPARISON ===

Plain Zarr (no CRS):
  _ARRAY_DIMENSIONS
  grid_mapping
  nodata

GeoZarr (with CRS):
  _ARRAY_DIMENSIONS
  grid_mapping
  nodata


## Step 6: Write GeoZarr to S3

In [8]:
os.environ["S3_BUCKET"] = "jmlane-demo-s3"
S3_BUCKET = os.environ.get("S3_BUCKET", "")

if not S3_BUCKET:
    print("S3_BUCKET not set — skipping S3 write.")
else:
    store_s3 = get_s3_store(S3_BUCKET, "sentinel2_geozarr.zarr")
    print(f"Writing GeoZarr to s3://{S3_BUCKET}/sentinel2_geozarr.zarr ...")
    ds_geo.to_zarr(store_s3, mode="w")
    print("Write complete.")

    ds_s3 = xr.open_zarr(store_s3, decode_coords="all")
    ds_s3 = ds_s3.rio.set_spatial_dims(x_dim="x", y_dim="y")
    print("CRS on S3 store:", ds_s3.rio.crs)

Writing GeoZarr to s3://jmlane-demo-s3/sentinel2_geozarr.zarr ...
Write complete.
CRS on S3 store: EPSG:32618


## Validation

In [9]:
assert os.path.isdir(ZARR_GEOZARR), "GeoZarr store not created"

z_check = zarr.open(ZARR_GEOZARR, mode="r")
red_check_attrs = dict(z_check["red"].attrs)
assert "grid_mapping" in red_check_attrs, \
    "Missing grid_mapping attribute — store is not GeoZarr compliant"

# Verify the spatial_ref coordinate exists with CRS info
assert "spatial_ref" in z_check, "Missing spatial_ref coordinate variable"
sr_check = dict(z_check["spatial_ref"].attrs)
assert "crs_wkt" in sr_check or "spatial_ref" in str(sr_check), \
    "spatial_ref variable missing CRS WKT"

# Verify xarray can re-read and resolve the CRS.
# decode_coords="all" promotes variables referenced by grid_mapping (like
# spatial_ref) from data variables to coordinates, which lets rioxarray
# find and parse the CRS WKT without calling write_crs() again.
ds_reread = xr.open_zarr(ZARR_GEOZARR, decode_coords="all") \
              .rio.set_spatial_dims(x_dim="x", y_dim="y")
assert ds_reread.rio.crs is not None, "xarray cannot resolve CRS from GeoZarr store"

print("All assertions passed.")
print(f"CRS resolved: {ds_reread.rio.crs}")

All assertions passed.
CRS resolved: EPSG:32618


---
## Learning Checkpoint — Q&A

**Q1:** What is the difference between the CRS stored in xarray's in-memory attributes vs. in Zarr's `.zattrs` on disk?

> *In-memory, the CRS lives in `DataArray.attrs` as a Python dict — it disappears when the process ends. In `.zattrs` on disk, it is persisted as JSON. The key distinction is that `.zattrs` can be read by any Zarr-aware tool (GDAL, QGIS, Julia, R) without needing xarray.*

**Q2:** What is a `grid_mapping` variable in CF conventions and why does GDAL require it?

> *CF conventions require that each georeferenced variable declare a `grid_mapping` attribute naming a scalar coordinate variable that holds the CRS definition. GDAL looks for this attribute to know which variable contains the projection — without it, GDAL treats the data as unprojected pixel coordinates.*

**Q3:** If you open a GeoZarr store in QGIS without any Python, how does QGIS know where the data is located on Earth?

> *QGIS uses GDAL's Zarr driver, which reads the `grid_mapping` attribute to find the `spatial_ref` variable, then parses its `crs_wkt` to construct a CRS. Combined with the x/y coordinate arrays (also in `.zattrs`), GDAL can compute the Affine transform and place pixels correctly on a map.*

**Q4:** What would break if you saved a Zarr store with `ds.to_zarr()` but skipped `rio.write_crs()` before saving?

> *The pixel values would be stored correctly, but no `grid_mapping` attribute or `spatial_ref` variable would exist. GDAL and QGIS would open the data with an unknown CRS and no georeferencing. The data would appear as a plain image with no location — unusable in any GIS context.*